In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

# 3 qubits, 3 classical bits
qc = QuantumCircuit(3, 3)

# Step 1: Prepare state to teleport → |1>
#qc.h(0)   # to prepare |+> state
qc.rx(0.7,0) # to prepare a state between |0> and |1> roatted by 0.7 radians around x-axis

qc.barrier()

# Step 2: Create Bell pair
qc.h(1)
qc.cx(1, 2)

qc.barrier()

# Step 3: Alice operations
qc.cx(0, 1)
qc.h(0)

qc.barrier()

# Step 4: Measure Alice
qc.measure(0, 0)
qc.measure(1, 1)

# Step 5: Conditional corrections
with qc.if_test((qc.clbits[1], 1)):
    qc.x(2)

with qc.if_test((qc.clbits[0], 1)):
    qc.z(2)

qc.barrier()

# Step 6: Measure Bob
qc.measure(2, 2)

print(qc.draw("text"))

# Run
sim = AerSimulator()

compiled = transpile(qc, sim)

result = sim.run(compiled, shots=1024).result()

counts = result.get_counts()

print(counts)

     ┌─────────┐ ░            ░      ┌───┐ ░ ┌─┐                             »
q_0: ┤ Rx(0.7) ├─░────────────░───■──┤ H ├─░─┤M├─────────────────────────────»
     └─────────┘ ░ ┌───┐      ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐                          »
q_1: ────────────░─┤ H ├──■───░─┤ X ├──────░──╫─┤M├──────────────────────────»
                 ░ └───┘┌─┴─┐ ░ └───┘      ░  ║ └╥┘  ┌──────  ┌───┐ ───────┐ »
q_2: ────────────░──────┤ X ├─░────────────░──╫──╫───┤ If-0  ─┤ X ├  End-0 ├─»
                 ░      └───┘ ░            ░  ║  ║   └──╥───  └───┘ ───────┘ »
                                              ║  ║ ┌────╨────┐               »
c: 3/═════════════════════════════════════════╩══╩═╡ c_1=0x1 ╞═══════════════»
                                              0  1 └─────────┘               »
«                                ░    
«q_0: ───────────────────────────░────
«                                ░    
«q_1: ───────────────────────────░────
«       ┌──────  ┌───┐ ───────┐  ░ ┌─┐
«q_2: ──┤ If-0 